In [40]:
import pandas as pd
from utils_io import salvar_df, carregar_df

In [24]:
df_politicas_base = carregar_df("blackjack_episodios_etapa_basica", pasta="dados")
df_qlearning = carregar_df("blackjack_resultados_qlearning", pasta="dados")

Lendo arquivo: C:\Users\Daianne\Documents\blackjack_RL\dados\blackjack_episodios_etapa_basica.csv
Lendo arquivo: C:\Users\Daianne\Documents\blackjack_RL\dados\blackjack_resultados_qlearning.csv


In [25]:
df_politicas_base.head()

,id_episodio,politica,resultado,recompensa,total_jogador,total_dealer,dealer_carta_visivel,qtd_passos,mao_jogador,mao_dealer
0,0,aleatoria,vitoria,1,17,26,10,1,"[7, 10]","[10, 5, 1, 10]"
1,1,aleatoria,derrota,-1,13,20,3,1,"[3, 10]","[3, 7, 10]"
2,2,aleatoria,vitoria,1,21,18,5,1,"[1, 10]","[5, 3, 10]"
3,3,aleatoria,vitoria,1,21,20,9,2,"[10, 4, 7]","[9, 2, 3, 6]"
4,4,aleatoria,derrota,-1,27,7,3,1,"[10, 7, 10]","[3, 4]"


In [26]:
df_qlearning.head()

,id_episodio,politica,resultado,recompensa,total_jogador,total_dealer,dealer_carta_visivel,qtd_passos,mao_jogador,mao_dealer
0,0,qlearning,empate,0,20,20,1,1,"[10, 10]","[1, 9]"
1,1,qlearning,derrota,-1,15,21,4,1,"[5, 10]","[4, 3, 4, 5, 5]"
2,2,qlearning,derrota,-1,18,21,10,1,"[10, 8]","[10, 3, 8]"
3,3,qlearning,derrota,-1,16,19,6,1,"[6, 10]","[6, 6, 7]"
4,4,qlearning,derrota,-1,18,21,10,1,"[8, 10]","[10, 1]"


In [27]:
colunas_padrao = [
    "id_episodio",
    "politica",
    "resultado",
    "recompensa",
    "total_jogador",
    "total_dealer",
    "dealer_carta_visivel",
    "qtd_passos",
    "mao_jogador",
    "mao_dealer"
]

df_politicas_base = df_politicas_base[colunas_padrao].copy()
df_qlearning = df_qlearning[colunas_padrao].copy()

In [28]:
df_politicas_base = df_politicas_base[
    df_politicas_base["politica"].isin(["aleatoria", "basica"])
].copy()

df_politicas_base["politica"].value_counts()

politica
aleatoria    5000
basica       5000
Name: count, dtype: int64

In [29]:
df_resultados_finais = pd.concat(
    [df_politicas_base, df_qlearning],
    ignore_index=True
)

df_resultados_finais.head()

,id_episodio,politica,resultado,recompensa,total_jogador,total_dealer,dealer_carta_visivel,qtd_passos,mao_jogador,mao_dealer
0,0,aleatoria,vitoria,1,17,26,10,1,"[7, 10]","[10, 5, 1, 10]"
1,1,aleatoria,derrota,-1,13,20,3,1,"[3, 10]","[3, 7, 10]"
2,2,aleatoria,vitoria,1,21,18,5,1,"[1, 10]","[5, 3, 10]"
3,3,aleatoria,vitoria,1,21,20,9,2,"[10, 4, 7]","[9, 2, 3, 6]"
4,4,aleatoria,derrota,-1,27,7,3,1,"[10, 7, 10]","[3, 4]"


In [30]:
df_resultados_finais.groupby("politica")["id_episodio"].count().reset_index(name="qtd_episodios")

,politica,qtd_episodios
0,aleatoria,5000
1,basica,5000
2,qlearning,5000


In [31]:
comparacao_contagem = (
    df_resultados_finais
    .groupby(["politica", "resultado"])
    .size()
    .reset_index(name="qtd")
)

total_por_politica = (
    df_resultados_finais
    .groupby("politica")
    .size()
    .reset_index(name="total_episodios")
)

df_comparacao_final = comparacao_contagem.merge(
    total_por_politica,
    on="politica",
    how="left"
)

df_comparacao_final["taxa_resultado"] = (
    df_comparacao_final["qtd"] / df_comparacao_final["total_episodios"]
)

df_comparacao_final = df_comparacao_final.sort_values(
    by=["politica", "resultado"]
).reset_index(drop=True)

df_comparacao_final

,politica,resultado,qtd,total_episodios,taxa_resultado
0,aleatoria,derrota,3227,5000,0.6454
1,aleatoria,empate,245,5000,0.0490
2,aleatoria,vitoria,1528,5000,0.3056
3,basica,derrota,2383,5000,0.4766
4,basica,empate,504,5000,0.1008
5,basica,vitoria,2113,5000,0.4226
6,qlearning,derrota,2400,5000,0.4800
7,qlearning,empate,448,5000,0.0896
8,qlearning,vitoria,2152,5000,0.4304


In [32]:
df_avaliacao_final = (
    df_resultados_finais
    .groupby("politica")
    .agg(
        episodios=("id_episodio", "count"),
        recompensa_media=("recompensa", "mean"),
        recompensa_total=("recompensa", "sum"),
        total_medio_jogador=("total_jogador", "mean"),
        total_medio_dealer=("total_dealer", "mean"),
        passos_medios=("qtd_passos", "mean")
    )
    .reset_index()
)

In [33]:
resumo_resultados = (
    df_resultados_finais
    .pivot_table(
        index="politica",
        columns="resultado",
        values="id_episodio",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

resumo_resultados.columns.name = None

if "vitoria" not in resumo_resultados.columns:
    resumo_resultados["vitoria"] = 0

if "derrota" not in resumo_resultados.columns:
    resumo_resultados["derrota"] = 0

if "empate" not in resumo_resultados.columns:
    resumo_resultados["empate"] = 0

df_avaliacao_final = df_avaliacao_final.merge(
    resumo_resultados[["politica", "vitoria", "derrota", "empate"]],
    on="politica",
    how="left"
)

df_avaliacao_final["taxa_vitoria"] = df_avaliacao_final["vitoria"] / df_avaliacao_final["episodios"]
df_avaliacao_final["taxa_derrota"] = df_avaliacao_final["derrota"] / df_avaliacao_final["episodios"]
df_avaliacao_final["taxa_empate"] = df_avaliacao_final["empate"] / df_avaliacao_final["episodios"]

df_avaliacao_final

,politica,episodios,recompensa_media,recompensa_total,total_medio_jogador,total_medio_dealer,passos_medios,vitoria,derrota,empate,taxa_vitoria,taxa_derrota,taxa_empate
0,aleatoria,5000,-0.3398,-1699,18.3896,18.6306,1.3378,1528,3227,245,0.3056,0.6454,0.0490
1,basica,5000,-0.0540,-270,20.3196,18.7396,1.6134,2113,2383,504,0.4226,0.4766,0.1008
2,qlearning,5000,-0.0496,-248,18.9292,19.3342,1.5654,2152,2400,448,0.4304,0.4800,0.0896


In [34]:
taxas_estouro = (
    df_resultados_finais
    .assign(
        estouro_jogador=lambda x: (x["total_jogador"] > 21).astype(int),
        estouro_dealer=lambda x: (x["total_dealer"] > 21).astype(int)
    )
    .groupby("politica")
    .agg(
        taxa_estouro_jogador=("estouro_jogador", "mean"),
        taxa_estouro_dealer=("estouro_dealer", "mean")
    )
    .reset_index()
)

df_avaliacao_final = df_avaliacao_final.merge(
    taxas_estouro,
    on="politica",
    how="left"
)

df_avaliacao_final

,politica,episodios,recompensa_media,recompensa_total,total_medio_jogador,total_medio_dealer,passos_medios,vitoria,derrota,empate,taxa_vitoria,taxa_derrota,taxa_empate,taxa_estouro_jogador,taxa_estouro_dealer
0,aleatoria,5000,-0.3398,-1699,18.3896,18.6306,1.3378,1528,3227,245,0.3056,0.6454,0.0490,0.2886,0.1970
1,basica,5000,-0.0540,-270,20.3196,18.7396,1.6134,2113,2383,504,0.4226,0.4766,0.1008,0.2778,0.2112
2,qlearning,5000,-0.0496,-248,18.9292,19.3342,1.5654,2152,2400,448,0.4304,0.4800,0.0896,0.1904,0.2434


In [35]:
df_avaliacao_final = df_avaliacao_final.sort_values(
    by=["taxa_vitoria", "recompensa_media"],
    ascending=False
).reset_index(drop=True)

df_avaliacao_final

,politica,episodios,recompensa_media,recompensa_total,total_medio_jogador,total_medio_dealer,passos_medios,vitoria,derrota,empate,taxa_vitoria,taxa_derrota,taxa_empate,taxa_estouro_jogador,taxa_estouro_dealer
0,qlearning,5000,-0.0496,-248,18.9292,19.3342,1.5654,2152,2400,448,0.4304,0.4800,0.0896,0.1904,0.2434
1,basica,5000,-0.0540,-270,20.3196,18.7396,1.6134,2113,2383,504,0.4226,0.4766,0.1008,0.2778,0.2112
2,aleatoria,5000,-0.3398,-1699,18.3896,18.6306,1.3378,1528,3227,245,0.3056,0.6454,0.0490,0.2886,0.1970


In [36]:
df_avaliacao_final[[
    "politica",
    "episodios",
    "vitoria",
    "derrota",
    "empate",
    "taxa_vitoria",
    "taxa_derrota",
    "taxa_empate",
    "recompensa_media",
    "passos_medios"
]]

,politica,episodios,vitoria,derrota,empate,taxa_vitoria,taxa_derrota,taxa_empate,recompensa_media,passos_medios
0,qlearning,5000,2152,2400,448,0.4304,0.4800,0.0896,-0.0496,1.5654
1,basica,5000,2113,2383,504,0.4226,0.4766,0.1008,-0.0540,1.6134
2,aleatoria,5000,1528,3227,245,0.3056,0.6454,0.0490,-0.3398,1.3378


In [37]:
salvar_df(df_comparacao_final, "blackjack_comparacao_final", pasta="resultados")
salvar_df(df_avaliacao_final, "blackjack_avaliacao_final", pasta="resultados")

Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\resultados\blackjack_comparacao_final.csv
Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\resultados\blackjack_avaliacao_final.csv


In [38]:
df_comparacao_final.head()

,politica,resultado,qtd,total_episodios,taxa_resultado
0,aleatoria,derrota,3227,5000,0.6454
1,aleatoria,empate,245,5000,0.0490
2,aleatoria,vitoria,1528,5000,0.3056
3,basica,derrota,2383,5000,0.4766
4,basica,empate,504,5000,0.1008


In [39]:
df_avaliacao_final.head()

,politica,episodios,recompensa_media,recompensa_total,total_medio_jogador,total_medio_dealer,passos_medios,vitoria,derrota,empate,taxa_vitoria,taxa_derrota,taxa_empate,taxa_estouro_jogador,taxa_estouro_dealer
0,qlearning,5000,-0.0496,-248,18.9292,19.3342,1.5654,2152,2400,448,0.4304,0.4800,0.0896,0.1904,0.2434
1,basica,5000,-0.0540,-270,20.3196,18.7396,1.6134,2113,2383,504,0.4226,0.4766,0.1008,0.2778,0.2112
2,aleatoria,5000,-0.3398,-1699,18.3896,18.6306,1.3378,1528,3227,245,0.3056,0.6454,0.0490,0.2886,0.1970
